In [1]:
# ============================================================
# STEP 1 : Mount Google Drive
# ============================================================

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# STEP 2 : Create Project Folder Structure
# ============================================================

import os

PROJECT_ROOT = "/content/drive/MyDrive/EmergencyVoiceAI"

folders = [
    "dataset/raw",
    "dataset/processed",
    "dataset/metadata",
    "models",
    "checkpoints",
    "logs",
    "exports"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_ROOT, folder), exist_ok=True)

print("Project folders are ready!")

print("\nCreated folders:\n")

for folder in folders:
    print(os.path.join(PROJECT_ROOT, folder))

Project folders are ready!

Created folders:

/content/drive/MyDrive/EmergencyVoiceAI/dataset/raw
/content/drive/MyDrive/EmergencyVoiceAI/dataset/processed
/content/drive/MyDrive/EmergencyVoiceAI/dataset/metadata
/content/drive/MyDrive/EmergencyVoiceAI/models
/content/drive/MyDrive/EmergencyVoiceAI/checkpoints
/content/drive/MyDrive/EmergencyVoiceAI/logs
/content/drive/MyDrive/EmergencyVoiceAI/exports


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ============================================================
# STEP 3 : Install Required Libraries
# ============================================================

!pip install -q tensorflow tensorflow-datasets librosa soundfile scipy pandas matplotlib scikit-learn tqdm

In [5]:
# ============================================================
# STEP 4 : Import Required Libraries
# ============================================================

import os
import random
import shutil
import zipfile

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.model_selection import train_test_split

print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("Librosa    :", librosa.__version__)
print("\nAll libraries imported successfully.")

TensorFlow : 2.20.0
NumPy      : 2.0.2
Librosa    : 0.11.0

All libraries imported successfully.


In [6]:
# ============================================================
# STEP 5 : Configure Google Drive Project Paths
# ============================================================

from pathlib import Path

# Root project directory in Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/EmergencyVoiceAI")

# Dataset folders
RAW_DATA_DIR = PROJECT_DIR / "dataset" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "dataset" / "processed"
METADATA_DIR = PROJECT_DIR / "dataset" / "metadata"

# Training folders
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "logs"

# Create folders if they don't exist
for folder in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    METADATA_DIR,
    CHECKPOINT_DIR,
    MODEL_DIR,
    LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders are ready.\n")

print("Raw Dataset :", RAW_DATA_DIR)
print("Processed   :", PROCESSED_DATA_DIR)
print("Metadata    :", METADATA_DIR)
print("Checkpoints :", CHECKPOINT_DIR)
print("Models      :", MODEL_DIR)
print("Logs        :", LOG_DIR)

Project folders are ready.

Raw Dataset : /content/drive/MyDrive/EmergencyVoiceAI/dataset/raw
Processed   : /content/drive/MyDrive/EmergencyVoiceAI/dataset/processed
Metadata    : /content/drive/MyDrive/EmergencyVoiceAI/dataset/metadata
Checkpoints : /content/drive/MyDrive/EmergencyVoiceAI/checkpoints
Models      : /content/drive/MyDrive/EmergencyVoiceAI/models
Logs        : /content/drive/MyDrive/EmergencyVoiceAI/logs


In [7]:
# ============================================================
# STEP 6 : Dataset Configuration
# ============================================================

# Sample rate for all audio
SAMPLE_RATE = 16000

# Audio clip duration (seconds)
CLIP_DURATION = 1.0

# Number of samples in one audio clip
NUM_SAMPLES = int(SAMPLE_RATE * CLIP_DURATION)

# MFCC parameters
N_MFCC = 40
N_FFT = 1024
HOP_LENGTH = 512

# Random seed
RANDOM_SEED = 42

# Emergency keywords
EMERGENCY_WORDS = [
    "help",
    "fire",
    "police",
    "ambulance",
    "emergency",
    "doctor"
]

print("Dataset configuration loaded successfully.\n")

print(f"Sample Rate      : {SAMPLE_RATE} Hz")
print(f"Clip Duration    : {CLIP_DURATION} sec")
print(f"Samples / Clip   : {NUM_SAMPLES}")
print(f"MFCC Features    : {N_MFCC}")
print(f"FFT Size         : {N_FFT}")
print(f"Hop Length       : {HOP_LENGTH}")
print(f"Emergency Words  : {EMERGENCY_WORDS}")

Dataset configuration loaded successfully.

Sample Rate      : 16000 Hz
Clip Duration    : 1.0 sec
Samples / Clip   : 16000
MFCC Features    : 40
FFT Size         : 1024
Hop Length       : 512
Emergency Words  : ['help', 'fire', 'police', 'ambulance', 'emergency', 'doctor']


In [8]:
# ============================================================
# STEP 7 : DOWNLOAD GOOGLE SPEECH COMMANDS DATASET
# Project : Emergency Voice AI
# Storage : /content (Temporary)
# ============================================================

import os
import urllib.request
from pathlib import Path

print("=" * 70)
print("STEP 7 : DOWNLOAD DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Temporary Working Directory
# ------------------------------------------------------------

TEMP_ROOT = Path("/content/EmergencyVoiceAI")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

# Dataset Paths
DATASET_URL = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"

ARCHIVE_PATH = TEMP_ROOT / "speech_commands_v0.02.tar.gz"

print(f"Download Location : {ARCHIVE_PATH}")

# ------------------------------------------------------------
# Download Dataset
# ------------------------------------------------------------

if ARCHIVE_PATH.exists():
    print("\n✅ Dataset archive already exists.")
else:
    print("\nDownloading Google Speech Commands v2...")
    urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)
    print("✅ Download completed.")

# ------------------------------------------------------------
# Verify Download
# ------------------------------------------------------------

size_gb = ARCHIVE_PATH.stat().st_size / (1024**3)

print(f"\nArchive Size : {size_gb:.2f} GB")

if size_gb < 2:
    raise RuntimeError(
        "Downloaded archive appears incomplete. "
        "Please delete it and run this step again."
    )

print("\n✅ Dataset downloaded successfully.")
print("Ready for Step 8 : Extract Dataset")

STEP 7 : DOWNLOAD DATASET
Download Location : /content/EmergencyVoiceAI/speech_commands_v0.02.tar.gz

✅ Download completed.

Archive Size : 2.26 GB

✅ Dataset downloaded successfully.
Ready for Step 8 : Extract Dataset


In [9]:
# ============================================================
# STEP 8 : EXTRACT GOOGLE SPEECH COMMANDS DATASET
# Project : Emergency Voice AI
# Storage : /content (Temporary)
# ============================================================

import tarfile
from pathlib import Path

print("=" * 70)
print("STEP 8 : EXTRACT DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TEMP_ROOT = Path("/content/EmergencyVoiceAI")

ARCHIVE_PATH = TEMP_ROOT / "speech_commands_v0.02.tar.gz"

EXTRACT_PATH = TEMP_ROOT / "speech_commands"

# ------------------------------------------------------------
# Verify Archive
# ------------------------------------------------------------

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(
        f"Dataset archive not found:\n{ARCHIVE_PATH}\n"
        "Please execute Step 7 first."
    )

# ------------------------------------------------------------
# Extract
# ------------------------------------------------------------

if EXTRACT_PATH.exists() and any(EXTRACT_PATH.iterdir()):
    print("✅ Dataset already extracted.")
else:
    print("Extracting dataset... (This may take 5-15 minutes)\n")

    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(EXTRACT_PATH)

    print("\n✅ Extraction completed successfully.")

# ------------------------------------------------------------
# Verify Extraction
# ------------------------------------------------------------

folders = [f for f in EXTRACT_PATH.iterdir() if f.is_dir()]

print(f"\nDataset Location : {EXTRACT_PATH}")
print(f"Total Class Folders : {len(folders)}")

print("\nReady for Step 9 : Verify Dataset")

STEP 8 : EXTRACT DATASET
Extracting dataset... (This may take 5-15 minutes)



/tmp/ipykernel_2012/2805500765.py:44: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(EXTRACT_PATH)



✅ Extraction completed successfully.

Dataset Location : /content/EmergencyVoiceAI/speech_commands
Total Class Folders : 36

Ready for Step 9 : Verify Dataset


In [10]:
# ============================================================
# STEP 9 : VERIFY DATASET
# ============================================================

from pathlib import Path

print("=" * 70)
print("STEP 9 : VERIFY DATASET")
print("=" * 70)

DATASET_ROOT = Path("/content/EmergencyVoiceAI/speech_commands")

# Ignore TensorFlow metadata folders/files
IGNORE = {
    "_background_noise_",
    ".ipynb_checkpoints",
    "__MACOSX"
}

classes = sorted([
    folder.name
    for folder in DATASET_ROOT.iterdir()
    if folder.is_dir() and folder.name not in IGNORE
])

print(f"\nTotal Keyword Classes : {len(classes)}\n")

total_audio = 0

print("-" * 55)
print(f"{'Keyword':20s}{'Audio Files'}")
print("-" * 55)

for cls in classes:

    count = len(list((DATASET_ROOT / cls).glob("*.wav")))

    total_audio += count

    print(f"{cls:20s}{count}")

print("-" * 55)

noise_count = len(list((DATASET_ROOT / "_background_noise_").glob("*.wav")))

print(f"\nBackground Noise Files : {noise_count}")

print(f"\nTotal Keyword Audio Files : {total_audio}")

print("\nDataset verification completed successfully.")

STEP 9 : VERIFY DATASET

Total Keyword Classes : 35

-------------------------------------------------------
Keyword             Audio Files
-------------------------------------------------------
backward            1664
bed                 2014
bird                2064
cat                 2031
dog                 2128
down                3917
eight               3787
five                4052
follow              1579
forward             1557
four                3728
go                  3880
happy               2054
house               2113
learn               1575
left                3801
marvin              2100
nine                3934
no                  3941
off                 3745
on                  3845
one                 3890
right               3778
seven               3998
sheila              2022
six                 3860
stop                3872
three               3727
tree                1759
two                 3880
up                  3723
visual              1592
wow

In [17]:
# ============================================================
# STEP 10 : PROJECT CONFIGURATION
# Project : Emergency Voice AI
# Hardware : ESP32-S3 N16R8 + INMP441
# ============================================================

from pathlib import Path

print("=" * 70)
print("STEP 10 : PROJECT CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Google Drive Project Folder
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/EmergencyVoiceAI")

# ------------------------------------------------------------
# Dataset Path (/content)
# ------------------------------------------------------------

RAW_DATASET_PATH = Path("/content/EmergencyVoiceAI/speech_commands")

# ------------------------------------------------------------
# Google Drive Output Folders
# ------------------------------------------------------------

PROCESSED_DIR = PROJECT_DIR / "dataset" / "processed"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
MODEL_DIR = PROJECT_DIR / "models"
EXPORT_DIR = PROJECT_DIR / "exports"
LOG_DIR = PROJECT_DIR / "logs"

for folder in [
    PROCESSED_DIR,
    CHECKPOINT_DIR,
    MODEL_DIR,
    EXPORT_DIR,
    LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Audio Configuration
# ------------------------------------------------------------

SAMPLE_RATE = 16000
CLIP_DURATION = 1.0
NUM_SAMPLES = int(SAMPLE_RATE * CLIP_DURATION)

# ------------------------------------------------------------
# Mel Spectrogram Configuration
# ------------------------------------------------------------

FRAME_LENGTH = 640
FRAME_STEP = 320
FFT_LENGTH = 1024
NUM_MEL_BINS = 64
LOWER_FREQ = 20
UPPER_FREQ = 4000

# ------------------------------------------------------------
# Training Configuration
# ------------------------------------------------------------

BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001

VALIDATION_SPLIT = 0.15
TEST_SPLIT = 0.15

RANDOM_SEED = 42

# ------------------------------------------------------------
# Display Configuration
# ------------------------------------------------------------

print("\nProject Directory")
print(PROJECT_DIR)

print("\nDataset")
print(RAW_DATASET_PATH)

print("\nAudio")
print(f"Sample Rate      : {SAMPLE_RATE}")
print(f"Clip Duration    : {CLIP_DURATION}")
print(f"Samples / Clip   : {NUM_SAMPLES}")

print("\nMel Spectrogram")
print(f"Mel Bands        : {NUM_MEL_BINS}")
print(f"FFT Length       : {FFT_LENGTH}")
print(f"Frame Length     : {FRAME_LENGTH}")
print(f"Frame Step       : {FRAME_STEP}")

print("\nTraining")
print(f"Epochs           : {EPOCHS}")
print(f"Batch Size       : {BATCH_SIZE}")
print(f"Learning Rate    : {LEARNING_RATE}")

print("\n✅ Configuration Loaded Successfully.")

STEP 10 : PROJECT CONFIGURATION

Project Directory
/content/drive/MyDrive/EmergencyVoiceAI

Dataset
/content/EmergencyVoiceAI/speech_commands

Audio
Sample Rate      : 16000
Clip Duration    : 1.0
Samples / Clip   : 16000

Mel Spectrogram
Mel Bands        : 64
FFT Length       : 1024
Frame Length     : 640
Frame Step       : 320

Training
Epochs           : 10
Batch Size       : 64
Learning Rate    : 0.001

✅ Configuration Loaded Successfully.


In [18]:
# ============================================================
# STEP 11 : LOAD DATASET & CREATE TENSORFLOW DATASET
# Project : Emergency Voice AI
# ============================================================

import tensorflow as tf
from pathlib import Path
import random

print("=" * 70)
print("STEP 11 : LOAD DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Verify Dataset Path
# ------------------------------------------------------------

if not RAW_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{RAW_DATASET_PATH}"
    )

# ------------------------------------------------------------
# Ignore these folders
# ------------------------------------------------------------

IGNORE_FOLDERS = {
    "_background_noise_",
    ".ipynb_checkpoints",
    "__MACOSX"
}

# ------------------------------------------------------------
# Get Class Names
# ------------------------------------------------------------

CLASS_NAMES = sorted([
    folder.name
    for folder in RAW_DATASET_PATH.iterdir()
    if folder.is_dir() and folder.name not in IGNORE_FOLDERS
])

NUM_CLASSES = len(CLASS_NAMES)

print(f"Total Classes : {NUM_CLASSES}")

# ------------------------------------------------------------
# Collect Audio Files
# ------------------------------------------------------------

audio_files = []
labels = []

for label_index, class_name in enumerate(CLASS_NAMES):

    wav_files = list((RAW_DATASET_PATH / class_name).glob("*.wav"))

    for wav in wav_files:
        audio_files.append(str(wav))
        labels.append(label_index)

# ------------------------------------------------------------
# Shuffle Dataset
# ------------------------------------------------------------

combined = list(zip(audio_files, labels))

random.seed(RANDOM_SEED)
random.shuffle(combined)

audio_files, labels = zip(*combined)

audio_files = list(audio_files)
labels = list(labels)

TOTAL_SAMPLES = len(audio_files)

print(f"Total Audio Files : {TOTAL_SAMPLES}")

# ------------------------------------------------------------
# Create TensorFlow Dataset
# ------------------------------------------------------------

dataset = tf.data.Dataset.from_tensor_slices(
    (audio_files, labels)
)

print("\nTensorFlow Dataset Created Successfully.")

print("\nFirst Five Classes")

for i, cls in enumerate(CLASS_NAMES[:5]):
    print(f"{i} -> {cls}")

print("\nReady for Step 12.")

STEP 11 : LOAD DATASET
Total Classes : 35
Total Audio Files : 105829

TensorFlow Dataset Created Successfully.

First Five Classes
0 -> backward
1 -> bed
2 -> bird
3 -> cat
4 -> dog

Ready for Step 12.


In [19]:
# ============================================================
# STEP 12 : AUDIO LOADING & PREPROCESSING
# Project : Emergency Voice AI
# ============================================================

import tensorflow as tf

print("=" * 70)
print("STEP 12 : AUDIO PREPROCESSING")
print("=" * 70)

# ------------------------------------------------------------
# Function : Read WAV File
# ------------------------------------------------------------

def load_audio(file_path, label):

    # Read audio file
    audio = tf.io.read_file(file_path)

    # Decode WAV
    audio, sample_rate = tf.audio.decode_wav(
        audio,
        desired_channels=1,
        desired_samples=NUM_SAMPLES
    )

    # Remove channel dimension
    audio = tf.squeeze(audio, axis=-1)

    # Normalize length
    audio = audio[:NUM_SAMPLES]

    padding = NUM_SAMPLES - tf.shape(audio)[0]

    audio = tf.pad(audio, [[0, padding]])

    return audio, label

# ------------------------------------------------------------
# Apply preprocessing
# ------------------------------------------------------------

dataset = dataset.map(
    load_audio,
    num_parallel_calls=tf.data.AUTOTUNE
)

print("✅ Audio preprocessing pipeline created.")

# ------------------------------------------------------------
# Verify One Sample
# ------------------------------------------------------------

for audio, label in dataset.take(1):

    print("\nAudio Shape :", audio.shape)

    print("Label Index :", label.numpy())

    print("Duration    :", audio.shape[0] / SAMPLE_RATE, "seconds")

print("\nReady for Step 13.")

STEP 12 : AUDIO PREPROCESSING
✅ Audio preprocessing pipeline created.

Audio Shape : (16000,)
Label Index : 14
Duration    : 1.0 seconds

Ready for Step 13.


In [20]:
# ============================================================
# STEP 13 : LOG-MEL SPECTROGRAM GENERATION
# Project : Emergency Voice AI
# ============================================================

import tensorflow as tf

print("=" * 70)
print("STEP 13 : LOG-MEL SPECTROGRAM")
print("=" * 70)

# ------------------------------------------------------------
# Create Mel Weight Matrix (only once)
# ------------------------------------------------------------

NUM_SPECTROGRAM_BINS = FFT_LENGTH // 2 + 1

mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
    num_mel_bins=NUM_MEL_BINS,
    num_spectrogram_bins=NUM_SPECTROGRAM_BINS,
    sample_rate=SAMPLE_RATE,
    lower_edge_hertz=LOWER_FREQ,
    upper_edge_hertz=UPPER_FREQ
)

# ------------------------------------------------------------
# Audio → Log-Mel Spectrogram
# ------------------------------------------------------------

def create_log_mel(audio, label):

    # Short-Time Fourier Transform
    stft = tf.signal.stft(
        audio,
        frame_length=FRAME_LENGTH,
        frame_step=FRAME_STEP,
        fft_length=FFT_LENGTH
    )

    # Magnitude Spectrogram
    spectrogram = tf.abs(stft)

    # Mel Spectrogram
    mel_spectrogram = tf.matmul(
        spectrogram,
        mel_weight_matrix
    )

    # Log-Mel Spectrogram
    log_mel = tf.math.log(
        mel_spectrogram + 1e-6
    )

    # Add channel dimension for CNN
    log_mel = tf.expand_dims(log_mel, axis=-1)

    return log_mel, label

# ------------------------------------------------------------
# Apply Feature Extraction
# ------------------------------------------------------------

feature_dataset = dataset.map(
    create_log_mel,
    num_parallel_calls=tf.data.AUTOTUNE
)

print("✅ Log-Mel feature pipeline created.")

# ------------------------------------------------------------
# Verify Output Shape
# ------------------------------------------------------------

for feature, label in feature_dataset.take(1):

    print("\nFeature Shape :", feature.shape)

    print("Label :", label.numpy())

print("\nReady for Step 14.")

STEP 13 : LOG-MEL SPECTROGRAM
✅ Log-Mel feature pipeline created.

Feature Shape : (49, 64, 1)
Label : 14

Ready for Step 14.


In [21]:
# ============================================================
# STEP 14 : CREATE TRAIN / VALIDATION / TEST DATASETS
# Project : Emergency Voice AI
# ============================================================

import tensorflow as tf

print("=" * 70)
print("STEP 14 : CREATE TRAIN / VALIDATION / TEST DATASETS")
print("=" * 70)

# ------------------------------------------------------------
# Total Samples
# ------------------------------------------------------------

TOTAL_SAMPLES = len(audio_files)

TRAIN_SIZE = int(TOTAL_SAMPLES * 0.70)
VALIDATION_SIZE = int(TOTAL_SAMPLES * 0.15)
TEST_SIZE = TOTAL_SAMPLES - TRAIN_SIZE - VALIDATION_SIZE

print(f"Total Samples      : {TOTAL_SAMPLES}")
print(f"Training Samples   : {TRAIN_SIZE}")
print(f"Validation Samples : {VALIDATION_SIZE}")
print(f"Testing Samples    : {TEST_SIZE}")

# ------------------------------------------------------------
# Split Dataset
# ------------------------------------------------------------

train_dataset = feature_dataset.take(TRAIN_SIZE)

remaining_dataset = feature_dataset.skip(TRAIN_SIZE)

validation_dataset = remaining_dataset.take(VALIDATION_SIZE)

test_dataset = remaining_dataset.skip(VALIDATION_SIZE)

# ------------------------------------------------------------
# Performance Optimizations
# ------------------------------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = (
    train_dataset
    .shuffle(5000, seed=RANDOM_SEED)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

validation_dataset = (
    validation_dataset
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("\n✅ Dataset split completed successfully.")

print("\nDataset Summary")

print("Training Batches   :", tf.data.experimental.cardinality(train_dataset).numpy())
print("Validation Batches :", tf.data.experimental.cardinality(validation_dataset).numpy())
print("Testing Batches    :", tf.data.experimental.cardinality(test_dataset).numpy())

print("\nReady for Step 15 : Build DS-CNN Model")

STEP 14 : CREATE TRAIN / VALIDATION / TEST DATASETS
Total Samples      : 105829
Training Samples   : 74080
Validation Samples : 15874
Testing Samples    : 15875

✅ Dataset split completed successfully.

Dataset Summary
Training Batches   : 1158
Validation Batches : 249
Testing Batches    : 249

Ready for Step 15 : Build DS-CNN Model


In [22]:
# ============================================================
# STEP 15 : BUILD KEYWORD SPOTTING CNN
# Project : Emergency Voice AI
# Target : ESP32-S3 N16R8
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, models

print("=" * 70)
print("STEP 15 : BUILD CNN MODEL")
print("=" * 70)

# ------------------------------------------------------------
# Input Shape
# ------------------------------------------------------------

INPUT_SHAPE = (49, 64, 1)

# ------------------------------------------------------------
# Build Model
# ------------------------------------------------------------

model = models.Sequential([

    # Input
    layers.Input(shape=INPUT_SHAPE),

    # Block 1
    layers.Conv2D(
        16,
        (3,3),
        padding="same",
        activation="relu"
    ),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    # Block 2
    layers.Conv2D(
        32,
        (3,3),
        padding="same",
        activation="relu"
    ),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    # Block 3
    layers.Conv2D(
        64,
        (3,3),
        padding="same",
        activation="relu"
    ),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    # Global Pooling
    layers.GlobalAveragePooling2D(),

    # Dropout
    layers.Dropout(0.30),

    # Output
    layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

# ------------------------------------------------------------
# Compile Model
# ------------------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ------------------------------------------------------------
# Model Summary
# ------------------------------------------------------------

model.summary()

print("\n✅ CNN Model Created Successfully.")

STEP 15 : BUILD CNN MODEL


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 49, 64, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 49, 64, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 24, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 24, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 24, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 12, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 12, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 35)             │         2,275 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,019 (101.64 KB)

 Trainable params: 25,795 (100.76 KB)

 Non-trainable params: 224 (896.00 B)


✅ CNN Model Created Successfully.


In [23]:
# ============================================================
# STEP 16 : TRAIN MODEL
# Project : Emergency Voice AI
# ============================================================

import os
import gc
import time
import tensorflow as tf
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger,
    BackupAndRestore,
    TensorBoard
)

print("=" * 70)
print("STEP 16 : TRAIN MODEL")
print("=" * 70)

# ------------------------------------------------------------
# Create Google Drive folders
# ------------------------------------------------------------

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# File Paths
# ------------------------------------------------------------

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_model.keras"
FINAL_MODEL_PATH = MODEL_DIR / "final_model.keras"
CSV_LOG_PATH = LOG_DIR / "training_history.csv"
TENSORBOARD_PATH = LOG_DIR / "tensorboard"

# ------------------------------------------------------------
# Callbacks
# ------------------------------------------------------------

checkpoint_callback = ModelCheckpoint(
    filepath=str(BEST_MODEL_PATH),
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=False,
    mode="max",
    verbose=1
)

backup_callback = BackupAndRestore(
    backup_dir=str(CHECKPOINT_DIR / "backup")
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    verbose=1
)

csv_logger = CSVLogger(
    str(CSV_LOG_PATH)
)

tensorboard = TensorBoard(
    log_dir=str(TENSORBOARD_PATH)
)

# ------------------------------------------------------------
# Train Model
# ------------------------------------------------------------

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=[
        checkpoint_callback,
        backup_callback,
        early_stop,
        reduce_lr,
        csv_logger,
        tensorboard
    ]
)

# ------------------------------------------------------------
# Save Final Model
# ------------------------------------------------------------

model.save(str(FINAL_MODEL_PATH))

# ------------------------------------------------------------
# Force Google Drive Sync
# ------------------------------------------------------------

os.sync()
gc.collect()
time.sleep(15)

# ------------------------------------------------------------
# Verify Saved Files
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VERIFYING SAVED FILES")
print("=" * 70)

files_to_check = [
    BEST_MODEL_PATH,
    FINAL_MODEL_PATH,
    CSV_LOG_PATH
]

all_saved = True

for file_path in files_to_check:

    if os.path.exists(str(file_path)):

        size = os.path.getsize(str(file_path)) / (1024 * 1024)

        print(f"✅ {file_path.name}")
        print(f"   Location : {file_path}")
        print(f"   Size     : {size:.2f} MB\n")

    else:

        print(f"❌ Missing : {file_path}\n")
        all_saved = False

# ------------------------------------------------------------
# Training Summary
# ------------------------------------------------------------

print("=" * 70)

if all_saved:

    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("✅ ALL IMPORTANT FILES ARE SAVED IN GOOGLE DRIVE")

else:

    raise FileNotFoundError(
        "\nERROR: One or more model files were NOT saved.\n"
        "Please DO NOT continue to Step 17."
    )

print("=" * 70)

print("\nFiles Saved")

print(BEST_MODEL_PATH)
print(FINAL_MODEL_PATH)
print(CSV_LOG_PATH)
print(TENSORBOARD_PATH)

# ------------------------------------------------------------
# Show Saved Keras Files
# ------------------------------------------------------------

print("\nSearching for saved .keras files...\n")

keras_found = False

for root, dirs, files in os.walk(str(PROJECT_DIR)):
    for file in files:
        if file.endswith(".keras"):
            keras_found = True
            print(os.path.join(root, file))

if not keras_found:
    print("❌ No .keras files found.")

# ------------------------------------------------------------
# Final Google Drive Check
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL GOOGLE DRIVE CHECK")
print("=" * 70)

print("\nCheckpoint Folder:")
print(os.listdir(CHECKPOINT_DIR))

print("\nModel Folder:")
print(os.listdir(MODEL_DIR))

print("\nLog Folder:")
print(os.listdir(LOG_DIR))

print("\n" + "=" * 70)
print("STEP 16 FINISHED")
print("=" * 70)

STEP 16 : TRAIN MODEL
Epoch 1/10
1158/1158 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - accuracy: 0.2187 - loss: 2.9403
Epoch 1: val_accuracy improved from None to 0.62889, saving model to /content/drive/MyDrive/EmergencyVoiceAI/checkpoints/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/EmergencyVoiceAI/checkpoints/best_model.keras
1158/1158 ━━━━━━━━━━━━━━━━━━━━ 487s 409ms/step - accuracy: 0.3769 - loss: 2.3474 - val_accuracy: 0.6289 - val_loss: 1.4190 - learning_rate: 0.0010
Epoch 2/10
1158/1158 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.6471 - loss: 1.3070
Epoch 2: val_accuracy improved from 0.62889 to 0.75539, saving model to /content/drive/MyDrive/EmergencyVoiceAI/checkpoints/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/EmergencyVoiceAI/checkpoints/best_model.keras
1158/1158 ━━━━━━━━━━━━━━━━━━━━ 458s 392ms/step - accuracy: 0.6768 - loss: 1.1795 - val_accuracy: 0.7554 - val_loss: 0.8760 - learning_rate: 0.0010
Epoch 3/10
1158/1

In [24]:
# ============================================================
# STEP 17A : VERIFY SAVED FILES IN GOOGLE DRIVE
# ============================================================

from pathlib import Path

print("=" * 70)
print("STEP 17A : VERIFY GOOGLE DRIVE FILES")
print("=" * 70)

PROJECT_DIR = Path("/content/drive/MyDrive/EmergencyVoiceAI")

required_files = [
    PROJECT_DIR / "checkpoints" / "best_model.keras",
    PROJECT_DIR / "models" / "final_model.keras",
    PROJECT_DIR / "logs" / "training_history.csv"
]

all_found = True

for file in required_files:
    if file.exists():
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"✅ {file.name:<25} Found ({size_mb:.2f} MB)")
    else:
        print(f"❌ {file.name:<25} NOT FOUND")
        all_found = False

print("=" * 70)

if all_found:
    print("✅ All important files are safely stored in Google Drive.")
    print("You can continue even if Colab disconnects.")
else:
    print("❌ Some required files are missing.")

STEP 17A : VERIFY GOOGLE DRIVE FILES
✅ best_model.keras          Found (0.35 MB)
✅ final_model.keras         Found (0.35 MB)
✅ training_history.csv      Found (0.00 MB)
✅ All important files are safely stored in Google Drive.
You can continue even if Colab disconnects.


In [25]:
# ============================================================
# STEP 17B : EVALUATE TRAINED MODEL
# ============================================================

from tensorflow import keras

print("=" * 70)
print("STEP 17B : MODEL EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Load model from Google Drive
# ------------------------------------------------------------

model = keras.models.load_model(
    MODEL_DIR / "final_model.keras"
)

print("✅ Model loaded successfully.")

# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

loss, accuracy = model.evaluate(
    test_dataset,
    verbose=1
)

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(f"Test Loss     : {loss:.4f}")
print(f"Test Accuracy : {accuracy*100:.2f}%")

print("\n✅ Step 17 completed successfully.")

STEP 17B : MODEL EVALUATION
✅ Model loaded successfully.
249/249 ━━━━━━━━━━━━━━━━━━━━ 120s 156ms/step - accuracy: 0.8450 - loss: 0.5285

FINAL TEST RESULTS
Test Loss     : 0.5285
Test Accuracy : 84.50%

✅ Step 17 completed successfully.


In [26]:
# ============================================================
# STEP 18 : CONVERT TO TENSORFLOW LITE (FLOAT32)
# Project : Emergency Voice AI
# ============================================================

import tensorflow as tf
from pathlib import Path

print("=" * 70)
print("STEP 18 : CONVERT TO TENSORFLOW LITE (FLOAT32)")
print("=" * 70)

# ------------------------------------------------------------
# Google Drive Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/EmergencyVoiceAI")

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load Trained Model
# ------------------------------------------------------------

keras_model_path = MODEL_DIR / "final_model.keras"

model = tf.keras.models.load_model(keras_model_path)

print("✅ Keras model loaded successfully.")

# ------------------------------------------------------------
# Convert to TensorFlow Lite
# ------------------------------------------------------------

converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

# ------------------------------------------------------------
# Save Model
# ------------------------------------------------------------

float_model_path = MODEL_DIR / "emergency_voice_float32.tflite"

with open(float_model_path, "wb") as f:
    f.write(tflite_model)

# ------------------------------------------------------------
# Display Information
# ------------------------------------------------------------

size_kb = float_model_path.stat().st_size / 1024

print("\n" + "=" * 70)
print("TensorFlow Lite Model Saved Successfully")
print("=" * 70)

print(f"Model Path : {float_model_path}")
print(f"Model Size : {size_kb:.2f} KB")

print("\n✅ Ready for Step 19 : INT8 Quantization")

STEP 18 : CONVERT TO TENSORFLOW LITE (FLOAT32)
✅ Keras model loaded successfully.
Saved artifact at '/tmp/tmp7txvkup2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 64, 1), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 35), dtype=tf.float32, name=None)
Captures:
  138699780926224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699780929104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699780929488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699780929680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699780929296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699780928336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699780928720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699780925456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699780928144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13

In [27]:
# ============================================================
# STEP 19 : FULL INT8 QUANTIZATION
# Project : Emergency Voice AI
# Target : ESP32-S3
# ============================================================

import tensorflow as tf
from pathlib import Path

print("=" * 70)
print("STEP 19 : FULL INT8 QUANTIZATION")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/EmergencyVoiceAI")
MODEL_DIR = PROJECT_DIR / "models"

keras_model_path = MODEL_DIR / "final_model.keras"

# ------------------------------------------------------------
# Load Model
# ------------------------------------------------------------

model = tf.keras.models.load_model(keras_model_path)

print("✅ Model loaded successfully.")

# ------------------------------------------------------------
# Representative Dataset
# ------------------------------------------------------------

def representative_dataset():

    for features, _ in train_dataset.take(100):
        for sample in features:
            yield [tf.expand_dims(sample, axis=0)]

# ------------------------------------------------------------
# TensorFlow Lite Converter
# ------------------------------------------------------------

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# ------------------------------------------------------------
# Convert
# ------------------------------------------------------------

tflite_quant_model = converter.convert()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

int8_model_path = MODEL_DIR / "emergency_voice_int8.tflite"

with open(int8_model_path, "wb") as f:
    f.write(tflite_quant_model)

# ------------------------------------------------------------
# Information
# ------------------------------------------------------------

size_kb = int8_model_path.stat().st_size / 1024

print("\n" + "=" * 70)
print("INT8 MODEL SAVED SUCCESSFULLY")
print("=" * 70)

print(f"Model Path : {int8_model_path}")
print(f"Model Size : {size_kb:.2f} KB")

print("\n✅ Ready for Step 20.")

STEP 19 : FULL INT8 QUANTIZATION
✅ Model loaded successfully.
Saved artifact at '/tmp/tmpq2iebnlw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 64, 1), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 35), dtype=tf.float32, name=None)
Captures:
  138699778557520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699778563664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138702474064336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138702474063952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138699778562512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138702474056848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138702474056656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138702474064912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138702474065104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138702474062800: Tenso

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 MODEL SAVED SUCCESSFULLY
Model Path : /content/drive/MyDrive/EmergencyVoiceAI/models/emergency_voice_int8.tflite
Model Size : 35.15 KB

✅ Ready for Step 20.


In [28]:
# ============================================================
# STEP 20 : VERIFY INT8 TENSORFLOW LITE MODEL
# Project : Emergency Voice AI
# ============================================================

import tensorflow as tf
import numpy as np
from pathlib import Path

print("=" * 70)
print("STEP 20 : VERIFY INT8 TENSORFLOW LITE MODEL")
print("=" * 70)

# ------------------------------------------------------------
# Load INT8 TensorFlow Lite Model
# ------------------------------------------------------------

MODEL_DIR = Path("/content/drive/MyDrive/EmergencyVoiceAI/models")

tflite_model_path = MODEL_DIR / "emergency_voice_int8.tflite"

interpreter = tf.lite.Interpreter(model_path=str(tflite_model_path))

interpreter.allocate_tensors()

print("✅ INT8 TensorFlow Lite model loaded successfully.")

# ------------------------------------------------------------
# Get Model Details
# ------------------------------------------------------------

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\nInput Details")
print("----------------------------------------")
print("Shape :", input_details[0]["shape"])
print("Dtype :", input_details[0]["dtype"])
print("Scale :", input_details[0]["quantization"][0])
print("Zero Point :", input_details[0]["quantization"][1])

print("\nOutput Details")
print("----------------------------------------")
print("Shape :", output_details[0]["shape"])
print("Dtype :", output_details[0]["dtype"])
print("Scale :", output_details[0]["quantization"][0])
print("Zero Point :", output_details[0]["quantization"][1])

# ------------------------------------------------------------
# Test One Sample
# ------------------------------------------------------------

for features, label in test_dataset.take(1):

    sample = features[0:1]

    break

# Quantize Input

input_scale, input_zero = input_details[0]["quantization"]

sample = sample.numpy()

sample_int8 = (sample / input_scale + input_zero).astype(np.int8)

# Run Inference

interpreter.set_tensor(
    input_details[0]["index"],
    sample_int8
)

interpreter.invoke()

output = interpreter.get_tensor(
    output_details[0]["index"]
)

prediction = np.argmax(output)

print("\nSample Label      :", int(label[0]))
print("Predicted Label   :", prediction)

print("\n✅ INT8 TensorFlow Lite model verified successfully.")
print("\nReady for Step 21 : Generate ESP32 Deployment Files")

STEP 20 : VERIFY INT8 TENSORFLOW LITE MODEL
✅ INT8 TensorFlow Lite model loaded successfully.

Input Details
----------------------------------------
Shape : [ 1 49 64  1]
Dtype : <class 'numpy.int8'>
Scale : 0.07784614711999893
Zero Point : 49

Output Details
----------------------------------------
Shape : [ 1 35]
Dtype : <class 'numpy.int8'>
Scale : 0.00390625
Zero Point : -128


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Sample Label      : 14
Predicted Label   : 14

✅ INT8 TensorFlow Lite model verified successfully.

Ready for Step 21 : Generate ESP32 Deployment Files


In [29]:
# ============================================================
# STEP 21A : VERIFY ALL DEPLOYMENT FILES
# ============================================================

from pathlib import Path

print("=" * 70)
print("STEP 21A : VERIFY DEPLOYMENT FILES")
print("=" * 70)

PROJECT_DIR = Path("/content/drive/MyDrive/EmergencyVoiceAI")
MODEL_DIR = PROJECT_DIR / "models"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
LOG_DIR = PROJECT_DIR / "logs"

files = [
    CHECKPOINT_DIR / "best_model.keras",
    MODEL_DIR / "final_model.keras",
    MODEL_DIR / "emergency_voice_float32.tflite",
    MODEL_DIR / "emergency_voice_int8.tflite",
    LOG_DIR / "training_history.csv",
]

all_ok = True

for file in files:
    if file.exists():
        size = file.stat().st_size / (1024 * 1024)
        print(f"✅ {file.name:<35} {size:.2f} MB")
    else:
        print(f"❌ {file.name:<35} Missing")
        all_ok = False

print("=" * 70)

if all_ok:
    print("✅ Everything required for deployment is safely stored in Google Drive.")
    print("You can continue even if the Colab runtime is reset.")
else:
    print("❌ Some deployment files are missing.")

STEP 21A : VERIFY DEPLOYMENT FILES
✅ best_model.keras                    0.35 MB
✅ final_model.keras                   0.35 MB
✅ emergency_voice_float32.tflite      0.10 MB
✅ emergency_voice_int8.tflite         0.03 MB
✅ training_history.csv                0.00 MB
✅ Everything required for deployment is safely stored in Google Drive.
You can continue even if the Colab runtime is reset.


In [30]:
# ============================================================
# STEP 22 : GENERATE DEPLOYMENT FILES
# Project : Emergency Voice AI
# ============================================================

import json
from pathlib import Path

print("=" * 70)
print("STEP 22 : GENERATE DEPLOYMENT FILES")
print("=" * 70)

# ------------------------------------------------------------
# Google Drive Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/EmergencyVoiceAI")

EXPORT_DIR = PROJECT_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. labels.txt
# ------------------------------------------------------------

labels_file = EXPORT_DIR / "labels.txt"

with open(labels_file, "w") as f:
    for label in CLASS_NAMES:
        f.write(label + "\n")

print("✅ labels.txt created")

# ------------------------------------------------------------
# 2. deployment_config.json
# ------------------------------------------------------------

config = {
    "model_name": "EmergencyVoiceAI",
    "model_type": "TensorFlow Lite INT8",
    "input_shape": [49, 64, 1],
    "num_classes": NUM_CLASSES,
    "sample_rate": SAMPLE_RATE,
    "clip_duration": CLIP_DURATION,
    "fft_length": FFT_LENGTH,
    "frame_length": FRAME_LENGTH,
    "frame_step": FRAME_STEP,
    "mel_bins": NUM_MEL_BINS,
    "model_file": "emergency_voice_int8.tflite",
    "board": "ESP32-S3 N16R8",
    "flash": "16MB",
    "psram": "8MB"
}

config_file = EXPORT_DIR / "deployment_config.json"

with open(config_file, "w") as f:
    json.dump(config, f, indent=4)

print("✅ deployment_config.json created")

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nDeployment Files")
print("-" * 50)
print(labels_file)
print(config_file)

print("\n✅ Step 22 completed successfully.")
print("All deployment metadata is now stored in Google Drive.")

STEP 22 : GENERATE DEPLOYMENT FILES
✅ labels.txt created
✅ deployment_config.json created

Deployment Files
--------------------------------------------------
/content/drive/MyDrive/EmergencyVoiceAI/exports/labels.txt
/content/drive/MyDrive/EmergencyVoiceAI/exports/deployment_config.json

✅ Step 22 completed successfully.
All deployment metadata is now stored in Google Drive.
